In [9]:
import os
from dotenv import load_dotenv

load_dotenv()  # This loads the variables from .env

key = os.getenv("OPEN_AI_KEY")



In [10]:
from openai import OpenAI
client = OpenAI(api_key=key)

response = client.embeddings.create(
    model="text-embedding-3-small",
    input="The quick brown fox jumped over the lazy dog"
)

embedding = response.data[0].embedding
print(len(embedding))  # e.g. 1536 dimensions

1536


In [11]:
import numpy as np
arr = np.load('output/embeddings_openai/text_embeddings.npy')
print(arr.shape)
print(arr.dtype)
print(arr[0][:10])

(200, 1536)
float32
[ 0.0312674   0.03370778  0.01938579  0.04606222 -0.00473968  0.03419586
  0.01772328 -0.01523714 -0.02811016 -0.0140627 ]


In [12]:
import faiss
index = faiss.read_index('output/embeddings_openai/text_faiss.index')
print('ntotal =', index.ntotal)
print('dim =', index.d)

ntotal = 200
dim = 1536


In [14]:

import json
import numpy as np

arr = np.load('output/embeddings_openai/text_embeddings.npy')
print('embedding shape:', arr.shape)

count = 0
with open('output/embeddings_openai/text_metadata.jsonl') as f:
    for _ in f:
        count += 1
print('metadata rows:', count)


embedding shape: (200, 1536)
metadata rows: 200


In [15]:
import numpy as np
arr = np.load('output/embeddings_openai/text_embeddings.npy')
norms = np.linalg.norm(arr, axis=1)
print('min norm:', norms.min())
print('max norm:', norms.max())
print('mean norm:', norms.mean())

min norm: 0.9999999
max norm: 1.0000001
mean norm: 1.0


The best quick check is:

pick one embedded chunk
search nearest neighbors
inspect whether the returned chunks are actually topically related


In [19]:
import json
from pathlib import Path

import faiss
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from sentence_transformers import SentenceTransformer

ROOT = Path.cwd()
ST_DIR = ROOT / "output" / "embeddings_sentencetransformers"
OPENAI_DIR = ROOT / "output" / "embeddings_openai"

TEST_QUERIES = [
    "bicycle bike crimes",
    "football game",
    "World War II campus",
    "divestment movement",
    "science library construction",
    "graduation ceremony",
    "university administration",
]
TOP_K = 3


def load_metadata(path: Path):
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle]


def preview_chunk(chunk_id: str, chunks_db: Path) -> str:
    import sqlite3

    conn = sqlite3.connect(chunks_db)
    try:
        row = conn.execute(
            "SELECT text FROM chunks WHERE chunk_id = ?",
            (chunk_id,),
        ).fetchone()
    finally:
        conn.close()
    if row is None:
        return "<missing chunk text>"
    return row[0][:220].replace("", " ")


def search(index_path: Path, metadata_path: Path, query_vector: np.ndarray, top_k: int):
    index = faiss.read_index(str(index_path))
    metadata = load_metadata(metadata_path)
    scores, neighbors = index.search(query_vector.astype("float32"), top_k)
    results = []
    for score, idx in zip(scores[0], neighbors[0]):
        row = metadata[idx]
        results.append({
            "score": float(score),
            **row,
        })
    return results


st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

load_dotenv(ROOT / ".env")
client = OpenAI(api_key=key)

chunks_db = ROOT / "output" / "metadata" / "chunks.db"

for query in TEST_QUERIES:
    print("=" * 100)
    print(f"QUERY: {query}")
    print("-" * 100)

    st_query = st_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    st_results = search(
        ST_DIR / "text_faiss.index",
        ST_DIR / "text_metadata.jsonl",
        st_query,
        TOP_K,
    )

    openai_query = client.embeddings.create(
        model="text-embedding-3-small",
        input=[query],
    )
    openai_vector = np.array([openai_query.data[0].embedding], dtype="float32")
    openai_vector = openai_vector / np.linalg.norm(openai_vector, axis=1, keepdims=True)
    openai_results = search(
        OPENAI_DIR / "text_faiss.index",
        OPENAI_DIR / "text_metadata.jsonl",
        openai_vector,
        TOP_K,
    )

    print("Sentence-transformers")
    for i, result in enumerate(st_results, start=1):
        print(f"  {i}. score={result['score']:.4f} | {result['date']} | {result['chunk_id']}")
        print("     ", preview_chunk(result["chunk_id"], chunks_db))

    print("OpenAI")
    for i, result in enumerate(openai_results, start=1):
        print(f"  {i}. score={result['score']:.4f} | {result['date']} | {result['chunk_id']}")
        print("     ", preview_chunk(result["chunk_id"], chunks_db))
    print()


/Users/rf50/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


QUERY: Robert Hutchins convocation
----------------------------------------------------------------------------------------------------
Sentence-transformers
  1. score=0.3933 | 1902-10-13 | mvol-0004-1902-1013::chunk-0018
       a n d   F r i d a y ,   I I   A .   M . - 1 2   M . ,   o f f i c e   o f   t h e   D i v i n i t y   D e a n s ,   r o o m   I S ,   H a s k e l l   m u s e u m .   S t u d e n t   A c t i v i t i e s   C h u r c h   H i s t o r y '   C l u b . - A d d r e s s   b y   P r o ­ f e r   H .   M :   S c o t t ,   o f   t h e   C h i c a g o   T h e o :   l o g i c a l   S e m m a r y .   S u b j e c t : "   C h u r c h   H i s ­ t o r y   
  2. score=0.3646 | 1902-10-15 | mvol-0004-1902-1015::chunk-0013
       T h e   s e n i o r   s t u d e n t s   o f   t h e   L a w   D e p a r t ­ m e n t ,   w h o   a r e   s t u d y i n g   f o r   t h e   d e g r e e   o f   L .   L .   B . ,   h e l d   a   s h o r t   c o n f e r e n c e   w i t h   D e a n   B e a l e  